# EvaLoop Assignment 4: code ↔ pseudocode loop on Kaggle (no API key)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**. Then click **Save Version → Save & Run All (Commit)** so the run continues with the browser closed. It takes roughly 3–4 h.

Runs 2 open models × {standard loop, code↔pseudocode loop} on all 378 MBPP+ tasks: greedy decoding, max 10 cycles, vLLM with each model's chat template, fp16.
Generated code runs inside this Kaggle container (sandboxed, away from your machine) through EvalPlus's `untrusted_check`.
Output: `/kaggle/working/evaloop_results.zip` (Output tab).

In [ ]:
import os
# ---- configuration ----
FORK_URL = "https://github.com/Arn15/EvaLooop.git"   # your public fork (branch main)
MODELS = ["Qwen/Qwen2.5-Coder-7B-Instruct", "ByteDance-Seed/Seed-Coder-8B-Instruct"]
LOOPS = ["standard", "code_pseudocode"]
N_TASKS = None        # None = all 378; e.g. 150 for a seeded subset if time is short
# T4 (sm75): vLLM 0.10.2 defaults to the slow FlexAttention kernel. We try vLLM's Triton
# kernel first (smoke test below) and fall back to FlexAttention if it fails on this GPU.
# (xFormers does not work here: its paged kernel needs 64-token KV pages, vLLM uses 16.)
BACKEND_CANDIDATES = ["TRITON_ATTN_VLLM_V1", "FLEX_ATTENTION"]
EXTRA_ARGS = "--gpu_memory_utilization 0.80 --max_num_seqs 96"   # avoids the Qwen CUDA OOM seen at 0.90

In [ ]:
import os, subprocess, glob
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Code: an uploaded Kaggle dataset containing EvaLooop-a4.tar.gz, else clone the fork
tars = glob.glob('/kaggle/input/**/EvaLooop-a4.tar.gz', recursive=True)
os.chdir('/kaggle/working')
if os.path.exists('EvaLooop'):
    pass
elif tars:
    !tar xzf {tars[0]} -C /kaggle/working
else:
    !git clone -q -b main {FORK_URL} EvaLooop
os.chdir('/kaggle/working/EvaLooop')
!git log --oneline | head -2
!ls scripts

In [ ]:
# Dependencies. vLLM is pinned to a CUDA-12.8 build (the latest vLLM pulls a CUDA-13 PyTorch
# that clashes with Kaggle's torchaudio), and transformers to 4.x (Kaggle ships 5.x, which
# removed tokenizer attributes vLLM 0.10.2 uses).
!nvidia-smi | head -4
!pip uninstall -y -q torchaudio
!pip install -q "vllm==0.10.2" "transformers==4.57.6" "huggingface-hub<1.0" "evalplus==0.3.1" fire tqdm matplotlib 2>&1 | grep -v "requires\|incompatible" | tail -3
!python -c "import torch, vllm, transformers; import transformers; print('transformers', transformers.__version__, '| torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu ok', torch.cuda.is_available(), '| vllm', vllm.__version__)"
# Download MBPP+ and compute the EvalPlus ground truth once
!python -W ignore -c "import sys; sys.path.insert(0,'.'); from evaloop.evaluation.py_testing import load_mbpp_dataset; p,_=load_mbpp_dataset(); print(len(p),'tasks')"
!python -m pytest -q tests 2>&1 | tail -2

In [ ]:
# ---- pick the fastest attention backend that works on this GPU (~2-3 min per try) ----
import subprocess, time, glob
SMOKE_MODEL = MODELS[0]
for backend in BACKEND_CANDIDATES:
    os.environ["VLLM_ATTENTION_BACKEND"] = backend
    t0 = time.time()
    r = subprocess.run(f"python scripts/run_loop.py --model {SMOKE_MODEL} --loop standard --backend vllm_chat "
                       f"--dtype half --tensor_parallel_size 2 --max_model_len 8192 {EXTRA_ARGS} "
                       f"--n_tasks 8 --max_cycles 1 --output_dir smoke_{backend}/ --log_level WARNING",
                       shell=True, capture_output=True, text=True)
    print(backend, "rc", r.returncode, f"{time.time()-t0:.0f}s", (r.stdout + r.stderr)[-600:], flush=True)
    if r.returncode == 0:
        print(">>> using attention backend", backend, flush=True)
        break
else:
    raise RuntimeError("no attention backend worked - see output above")

import time
subset = f"--n_tasks {N_TASKS} --sample_seed 0" if N_TASKS else ""
os.makedirs('logs', exist_ok=True)
for model in MODELS:
    for loop in LOOPS:
        alias = model.split('/')[-1].lower()
        if glob.glob(f"results/{alias}__{loop}_{loop}_results.json"):
            print('already done:', alias, loop); continue
        t0 = time.time()
        cmd = (f"python scripts/run_loop.py --model {model} --loop {loop} --backend vllm_chat "
               f"--dtype half --tensor_parallel_size 2 --max_model_len 8192 "
               f"{EXTRA_ARGS} {subset} --output_dir results/ --log_level WARNING")
        print('>>>', cmd, flush=True)
        with open(f"logs/{alias}__{loop}.log", "w") as log:
            rc = subprocess.run(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT).returncode
        print(open(f"logs/{alias}__{loop}.log").read()[-1500:])
        print(f"exit={rc}  minutes={(time.time()-t0)/60:.1f}", flush=True)
        if rc != 0:
            raise RuntimeError(f"{alias}/{loop} failed - see the log above; stopping so no GPU time is wasted")

In [ ]:
!python scripts/analyze_extension.py --results_dir results --out_dir analysis
!cd /kaggle/working/EvaLooop && zip -qr /kaggle/working/evaloop_results.zip results logs analysis
!ls -la /kaggle/working/evaloop_results.zip